In [1]:
import os
import re
import shutil
from pathlib import Path

# Define root paths relative to the notebook's location
DOCS_DIR = Path("docs")
IMAGES_DIR = DOCS_DIR / "assets" / "images"
BACKUP_DIR = Path("orphaned_images_backup")

# Regex pattern to match markdown image syntax: ![alt](path)
# This captures the alt text in group 1 and the image path/filename in group 2
IMG_REGEX = re.compile(r"!\[(.*?)\]\((.*?)\)")

In [2]:
def process_markdown_images(fix_paths=False):
    """
    Crawls markdown files to fix image paths to relative asset targets
    and detects broken/missing images.
    """
    missing_images = set()
    referenced_images = set()
    
    # Walk through all markdown files in the docs directory
    for md_path in DOCS_DIR.rglob("*.md"):
        # Skip assets or themes if any accidentally match
        if "assets" in md_path.parts:
            continue
            
        with open(md_path, "r", encoding="utf-8") as f:
            content = f.read()
            
        matches = IMG_REGEX.findall(content)
        if not matches:
            continue
            
        new_content = content
        file_updated = False
        
        for alt_text, img_path in matches:
            # Extract just the raw filename (e.g., 'page_4_image_1_v2.jpg')
            img_name = Path(img_path).name
            referenced_images.add(img_name)
            
            # Check if the file actually exists in assets/images
            actual_image_path = IMAGES_DIR / img_name
            if not actual_image_path.exists():
                missing_images.add((md_path.name, img_name))
            
            if fix_paths:
                # Calculate the relative path from the current MD file to assets/images
                # e.g., from 'docs/Unit 1/file.md' to 'docs/assets/images/' is '../assets/images/'
                relative_dir = os.path.relpath(IMAGES_DIR, md_path.parent)
                correct_img_path = os.path.join(relative_dir, img_name).replace("\\", "/")
                
                # Replace the old image reference with the clean relative one
                old_markdown_link = f"![{alt_text}]({img_path})"
                new_markdown_link = f"![{alt_text}]({correct_img_path})"
                
                if old_markdown_link != new_markdown_link:
                    new_content = new_content.replace(old_markdown_link, new_markdown_link)
                    file_updated = True
                    
        if fix_paths and file_updated:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(new_content)
            print(f"✔️ Updated paths in: {md_path.relative_to(DOCS_DIR)}")
            
    # Print results for broken links
    print("\n--- BROKEN / MISSING IMAGES REPORT ---")
    if missing_images:
        print(f"❌ Found {len(missing_images)} broken image link(s):")
        for md_file, missing_img in sorted(missing_images):
            print(f"   - In '{md_file}': Graphic '{missing_img}' is missing from assets folder.")
    else:
        print("✅ No broken image links detected! All referenced images exist.")
        
    return referenced_images

In [3]:
def clean_orphaned_images(referenced_images):
    """
    Compares physical files against referenced images and moves unused ones to a backup folder.
    """
    if not IMAGES_DIR.exists():
        print("Error: Images directory does not exist.")
        return

    # Gather all images currently inside the folder
    physical_images = {f.name for f in IMAGES_DIR.iterdir() if f.is_file()}
    
    # Orphaned images are those present physically but never linked in code
    orphaned_images = physical_images - referenced_images
    
    print("\n--- ORPHANED IMAGES REPORT ---")
    if orphaned_images:
        print(f"📦 Found {len(orphaned_images)} orphaned image(s). Moving to '{BACKUP_DIR}/'...")
        BACKUP_DIR.mkdir(exist_ok=True)
        
        for img_name in sorted(orphaned_images):
            source = IMAGES_DIR / img_name
            destination = BACKUP_DIR / img_name
            shutil.move(str(source), str(destination))
            print(f"   -> Moved: {img_name}")
        print("📁 Cleanup complete.")
    else:
        print("🎉 Zero orphaned images found! Your assets folder is pristine.")

In [8]:
used_images = process_markdown_images(fix_paths=True)
used_images

✔️ Updated paths in: Unit 1\1_1_describing-motion.md
✔️ Updated paths in: Unit 1\1_2_explaining-motion.md
✔️ Updated paths in: Unit 1\1_4_momentun-and-collisions.md
✔️ Updated paths in: Unit 1\1_5_stretching-effect-of-forces.md
✔️ Updated paths in: Unit 1\1_6_turning-effect-of-forces.md
✔️ Updated paths in: Unit 2\2_1_static-electricity.md
✔️ Updated paths in: Unit 2\2_2_electric-circuits.md
✔️ Updated paths in: Unit 2\2_3_mains-electricity.md
✔️ Updated paths in: Unit 3\3_1_describing-waves.md
✔️ Updated paths in: Unit 3\3_2_the-electromagnetic-spectrum.md
✔️ Updated paths in: Unit 3\3_3_light-waves.md
✔️ Updated paths in: Unit 3\3_4_sound-waves.md
✔️ Updated paths in: Unit 4\4_1_mechanical-energy-transfers.md
✔️ Updated paths in: Unit 4\4_2_energy_stores_and_transfers.md
✔️ Updated paths in: Unit 4\4_3_heat_transfers.md
✔️ Updated paths in: Unit 4\4_4_energy-resources.md
✔️ Updated paths in: Unit 5\5_1_density-and-pressure.md
✔️ Updated paths in: Unit 5\5_2_changes-of-state.md
✔️ Upd

{'image_url_placeholder',
 'page_100_image_2_v2.jpg',
 'page_101_image_2_v2.jpg',
 'page_101_image_8_v2.jpg',
 'page_104_chart_1_v2.jpg',
 'page_106_image_1_v2.jpg',
 'page_107_image_1_v2.jpg',
 'page_108_image_1_v2.jpg',
 'page_10_chart_1_v2.jpg',
 'page_10_image_3_v2.jpg',
 'page_110_image_1_v2.jpg',
 'page_110_image_3_v2.jpg',
 'page_111_image_1_v2.jpg',
 'page_111_image_2_v2.jpg',
 'page_112_image_1_v2.jpg',
 'page_114_chart_1_v2.jpg',
 'page_115_image_1_v2.jpg',
 'page_115_image_2_v2.jpg',
 'page_118_image_2_v2.jpg',
 'page_119_image_1_v2.jpg',
 'page_120_image_1_v2.jpg',
 'page_120_image_3_v2.jpg',
 'page_122_image_1_v2.jpg',
 'page_124_image_1_v2.jpg',
 'page_125_image_1_v2.jpg',
 'page_125_image_2_v2.jpg',
 'page_126_image_2_v2.jpg',
 'page_128_image_1_v2.jpg',
 'page_130_image_1_v2.jpg',
 'page_132_image_1_v2.jpg',
 'page_133_image_1_v2.jpg',
 'page_134_image_1_v2.jpg',
 'page_135_image_1_v2.jpg',
 'page_135_image_2_v2.jpg',
 'page_135_image_4_v2.jpg',
 'page_137_image_1_v2.jp

In [9]:
len(used_images)

397

In [10]:
clean_orphaned_images(used_images)


--- ORPHANED IMAGES REPORT ---
🎉 Zero orphaned images found! Your assets folder is pristine.
